# Grafico 04 - Anillo vs. buffer segun distancia al borde del AP

Este notebook reproduce en Google Colab las 2 imagenes de este grafico (general y por tipologia) y su tabla de soporte en Excel.

**Antes de correr las celdas de abajo**, ten a mano el archivo `naturalidad_data.json` (esta en la carpeta `codigo/` de este grafico, en tu computador o en tu repositorio de GitHub).

Corre las celdas en orden, de arriba hacia abajo.

## 1. Instalar paquetes

In [ ]:
!pip -q install numpy pandas matplotlib openpyxl


## 2. Subir el archivo de datos

In [ ]:
from google.colab import files
print("Sube aqui: naturalidad_data.json")
uploaded = files.upload()


## 3. Generar las 2 imagenes

In [ ]:
"""
Anillo vs. buffer — superficie natural según distancia al borde del AP
=========================================================================

Gráfico corregido según comentarios de Vale (agosto-2026): limpieza de
diseño y terminología, misma lógica de "general + por tipología" que el
resto de los gráficos de este proyecto. Ver README.txt y METODOLOGIA.docx
de esta carpeta para el detalle completo.

QUÉ HACE ESTE SCRIPT
--------------------
Genera 2 gráficos de líneas que comparan dos formas distintas de medir la
superficie natural alrededor de una AP, en el año 2024:

  - ANILLO   -> el % de superficie natural en la franja AISLADA a esa
                distancia (ej. "5km" = solo el anillo entre 4km y 5km del
                borde, sin contar lo que hay más cerca).
  - BUFFER   -> el % de superficie natural ACUMULADO desde el borde hasta
                esa distancia (ej. "5km" = todo el área entre el borde del
                AP y 5km, incluyendo los anillos de 1km a 5km juntos).

Los 2 gráficos son:
  1. anillo_vs_buffer_general.png         -> promedio nacional (97 AP)
  2. anillo_vs_buffer_por_tipologia.png   -> lo mismo, en 3 paneles (PN/RN/MN)

DISEÑO: solo título + nombres de ejes + el gráfico (más la leyenda
Anillo/Buffer, que es parte del gráfico) -- sin subtítulo ni notas al pie
sobre la imagen. Terminología corregida: "superficie natural", no
"cobertura natural".

ARCHIVO DE ENTRADA (debe estar en esta misma carpeta `codigo/`)
------------------------------------------------------------------
  naturalidad_data.json   -> trae, además de "anillos" (franja aislada),
                              la serie "buffer" (acumulado desde el borde)
                              para cada AP, año y distancia.

SALIDA (se guarda en ../imagenes/)
-----------------------------------
  anillo_vs_buffer_general.png
  anillo_vs_buffer_por_tipologia.png

Para correrlo: python3 anillo_vs_buffer.py
(requiere numpy, matplotlib -- instalar con: pip install numpy matplotlib)

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que las imágenes se guarden en otro lugar
     -> variable OUT_DIR, más abajo.
  ...cambia el año que se analiza (hoy: 2024)
     -> variable ANIO, más abajo.
  ...quieres cambiar tamaño de letra, colores, tamaño de figura, etc.
     (ajustes puramente visuales)
     -> están marcados con "<-- AJUSTE VISUAL" en cada sección.
===========================================================================
"""

import json
import re
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# CONFIGURACIÓN DE RUTAS
BASE_DIR = "/content"
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, "naturalidad_data.json")
OUT_DIR = os.path.join(BASE_DIR, "imagenes")
os.makedirs(OUT_DIR, exist_ok=True)

ANIO = 2024

# PALETA Y ESTILO
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"
CAT_BLUE = "#2a78d6"
CAT_ORANGE = "#eb6834"

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]

def style_ax(ax):
    ax.set_facecolor(SURFACE)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(BASELINE)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.xaxis.label.set_color(INK_SECONDARY)
    ax.yaxis.label.set_color(INK_SECONDARY)

# CARGA DE DATOS
DATA = json.load(open(NATURALIDAD_JSON_PATH))
DIST = DATA["dist"]
APS = DATA["aps"]
ANILLOS = DATA["anillos"]
BUFFER = DATA["buffer"]
x_order = DIST
xs = range(len(x_order))

def tipologia(nombre):
    m = re.match(r"^(MN|PN|RN)\s", nombre)
    return m.group(1) if m else "??"

TIPO_ORDER = ["PN", "RN", "MN"]
TIPO_NOMBRE = {"PN": "Parque Nacional", "RN": "Reserva Nacional", "MN": "Monumento Natural"}

for a in APS:
    a["tipo"] = tipologia(a["name"])

def ap_val(ap_name, year, dist_label, series):
    idx = DIST.index(dist_label)
    return series[ap_name][str(year)][idx]

def promedios(nombres_ap):
    anillo_mean, buffer_mean = [], []
    for d in x_order:
        a_vals = [ap_val(nm, ANIO, d, ANILLOS) for nm in nombres_ap]
        b_vals = [ap_val(nm, ANIO, d, BUFFER) for nm in nombres_ap]
        anillo_mean.append(np.nanmean([v for v in a_vals if v is not None]))
        buffer_mean.append(np.nanmean([v for v in b_vals if v is not None]))
    return anillo_mean, buffer_mean

# GRÁFICO GENERAL
anillo_mean, buffer_mean = promedios([a["name"] for a in APS])

fig, ax = plt.subplots(figsize=(7.5, 5.6), dpi=200)
fig.patch.set_facecolor(SURFACE)
style_ax(ax)
ax.fill_between(xs, anillo_mean, buffer_mean, color=CAT_ORANGE, alpha=0.15, zorder=1)
ax.plot(xs, anillo_mean, color=CAT_BLUE, linewidth=2.4, marker="o", markersize=5,
        label="Anillo (franja aislada)", zorder=3)
ax.plot(xs, buffer_mean, color=CAT_ORANGE, linewidth=2.4, marker="o", markersize=5,
        label="Buffer (acumulado)", zorder=3)
ax.set_xticks(xs)
ax.set_xticklabels(x_order)
ax.set_ylabel(f"% superficie natural (promedio nacional, {len(APS)} AP, {ANIO})")
ax.set_xlabel("Distancia desde el borde del AP")
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
leg = ax.legend(loc="upper right", frameon=False, fontsize=9)
for t in leg.get_texts():
    t.set_color(INK_SECONDARY)

fig.suptitle("Comparación entre anillos de distancia y buffers acumulados\nen la superficie natural alrededor del AP",
             color=INK_PRIMARY, fontsize=13.5, fontweight="bold", x=0.02, ha="left", y=0.995, va="top")
fig.subplots_adjust(top=0.83, bottom=0.11, left=0.11, right=0.97)
fig.savefig(os.path.join(OUT_DIR, "anillo_vs_buffer_general.png"), facecolor=SURFACE)
plt.close(fig)
print("OK anillo_vs_buffer_general.png")

# GRÁFICO POR TIPOLOGÍA
fig, axes = plt.subplots(1, 3, figsize=(13, 5), dpi=200, sharey=True)
fig.patch.set_facecolor(SURFACE)
for ax, t in zip(axes, TIPO_ORDER):
    style_ax(ax)
    aps_t = [a["name"] for a in APS if a["tipo"] == t]
    am, bm = promedios(aps_t)
    ax.fill_between(xs, am, bm, color=CAT_ORANGE, alpha=0.15, zorder=1)
    ax.plot(xs, am, color=CAT_BLUE, linewidth=2.2, marker="o", markersize=4.5, label="Anillo", zorder=3)
    ax.plot(xs, bm, color=CAT_ORANGE, linewidth=2.2, marker="o", markersize=4.5, label="Buffer", zorder=3)
    ax.set_xticks(xs)
    ax.set_xticklabels(x_order, fontsize=8, rotation=45, ha="right")
    ax.set_title(f"{TIPO_NOMBRE[t]} ({len(aps_t)} AP)", color=INK_PRIMARY, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Distancia desde el borde del AP", fontsize=9)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
axes[0].set_ylabel(f"% superficie natural ({ANIO})")
leg = axes[-1].legend(loc="upper right", frameon=False, fontsize=9)
for t in leg.get_texts():
    t.set_color(INK_SECONDARY)

fig.suptitle("Anillo vs. buffer por tipología de área protegida", color=INK_PRIMARY,
             fontsize=14.5, fontweight="bold", x=0.02, ha="left", y=0.995, va="top")
fig.subplots_adjust(top=0.84, bottom=0.19, left=0.06, right=0.97, wspace=0.1)
fig.savefig(os.path.join(OUT_DIR, "anillo_vs_buffer_por_tipologia.png"), facecolor=SURFACE)
plt.close(fig)
print("OK anillo_vs_buffer_por_tipologia.png")


## 4. Generar la tabla de soporte (Excel)

In [ ]:
import pandas as pd

OUT_XLSX = os.path.join(BASE_DIR, "tabla_soporte.xlsx")

def promedios_excel(nombres_ap):
    anillo_mean, buffer_mean = [], []
    for d in DIST:
        a_vals = [ap_val(nm, ANIO, d, ANILLOS) for nm in nombres_ap]
        b_vals = [ap_val(nm, ANIO, d, BUFFER) for nm in nombres_ap]
        anillo_mean.append(np.nanmean([v for v in a_vals if v is not None]))
        buffer_mean.append(np.nanmean([v for v in b_vals if v is not None]))
    return anillo_mean, buffer_mean

# HOJA 1: General por distancia
am, bm = promedios_excel([a["name"] for a in APS])
df_general = pd.DataFrame({
    "distancia": DIST,
    "anillo_pct_natural_promedio": [round(float(v), 2) for v in am],
    "buffer_pct_natural_promedio": [round(float(v), 2) for v in bm],
    "diferencia_buffer_menos_anillo": [round(float(b - a), 2) for a, b in zip(am, bm)],
})

# HOJA 2: Por tipología y distancia
rows = []
for t in TIPO_ORDER:
    aps_t = [a["name"] for a in APS if a["tipo"] == t]
    am_t, bm_t = promedios_excel(aps_t)
    for d, a_v, b_v in zip(DIST, am_t, bm_t):
        rows.append({
            "tipologia": t, "n_ap": len(aps_t), "distancia": d,
            "anillo_pct_natural_promedio": round(float(a_v), 2),
            "buffer_pct_natural_promedio": round(float(b_v), 2),
            "diferencia_buffer_menos_anillo": round(float(b_v - a_v), 2),
        })
df_tipo = pd.DataFrame(rows)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_general.to_excel(writer, sheet_name="general_por_distancia", index=False)
    df_tipo.to_excel(writer, sheet_name="por_tipologia_y_distancia", index=False)

print(f"OK {OUT_XLSX} -- 2 hojas")


## 5. Ver las imagenes generadas

In [ ]:
import glob
from IPython.display import Image, display

for p in sorted(glob.glob(os.path.join(OUT_DIR, '*.png'))):
    print(p.split('/')[-1])
    display(Image(filename=p))


## 6. Descargar todo (imagenes + tabla de soporte) en un .zip

In [ ]:
import shutil, os
from google.colab import files

RESULT_DIR = "/content/resultados_04_anillo_vs_buffer"
os.makedirs(RESULT_DIR, exist_ok=True)
if os.path.isdir(OUT_DIR):
    shutil.copytree(OUT_DIR, os.path.join(RESULT_DIR, "imagenes"), dirs_exist_ok=True)
if os.path.exists(OUT_XLSX):
    shutil.copy(OUT_XLSX, RESULT_DIR)
shutil.make_archive(RESULT_DIR, "zip", RESULT_DIR)
files.download(RESULT_DIR + ".zip")
print("Listo:", RESULT_DIR + ".zip")
